### Pronunciation

This week we are going to focus on using computational methods to get the pronunciation of a word.

Before we jump into programming, let's make sure we all know the basic terminologies here.

- Phoneme: the smallest unit of sound in a language, it can be a vowel or a consonant
- Syllabus: a "beat" in a word, containing a single vowel sound and often surrounding consonants. It can consist of multiple phonemes.

In the English language, most of the tools that are built around pronunciation is based on [the CMU Dictionary](http://www.speech.cs.cmu.edu/cgi-bin/cmudict). It is an open-source machine-readable pronunciation dictionary for North American English that contains over 134,000 words and their pronunciations.

It uses the [ARPAbet](https://en.wikipedia.org/wiki/ARPABET) to represent the phoneme, a phonetic standard developed for speech understanding/synthesizing. ARPAbet only uses ASCII characters to represent the phonemes, which makes it easier for a machine to process compared with the IPA(International Phonetic Alphabet) that are usually used in dictionaries. The [current phoneme set](http://www.speech.cs.cmu.edu/cgi-bin/cmudict#phones) contains 39 phonemes, from which vowels carry a lexical stress marker:
- 0    — No stress
- 1    — Primary stress
- 2    — Secondary stress

---

Let's first install the python library, `pronouncing`.

In [1]:
!pip install pronouncing

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 25.6 MB/s  0:00:00
  Created wheel for pronouncing: filename=pronouncing-0.2.0-py2.py3-none-any.whl size=6340 sha256=820e0485b59e0d1fe1e8e74b99e61e4f9fc118aede3caa3c2f532676699f4071
  Stored in directory: /Users/cqx931/Library/Caches/pip/wheels/a0/76/15/dfdf38731993cdc4e86fd6d949c70c0e9786cf00073d8114d4
Successfully built pronouncing
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pronouncing]


In [3]:
import pronouncing

The function `pronouncing.phones_for_word()` returns a list of all pronunciations for the given word found in the CMU pronouncing dictionary. Each token in a pronunciation string is called a “phone.”

In [13]:
pronouncing.phones_for_word("cat")

['K AE1 T']

Sometimes, the pronouncing dictionary has more than one pronunciation for the same word. “Permit” is a good example: it can be pronounced either with the stress on the first syllable (as a noun, “do you have a permit to enter?”) or on the second syllable (as a verb, “will you permit me to stay here?”).

In [16]:
pronouncing.phones_for_word("permit")

['P ER0 M IH1 T', 'P ER1 M IH2 T']

After getting the phones of a word, we can pass it further to `pronouncing.syllable_count()` to count the number of syllables.

In [19]:
phones_list = pronouncing.phones_for_word("coconut")
pronouncing.syllable_count(phones_list[0])

3

We can even use it to count the number of syllables for a sentence

In [20]:
text = "Thou shalt not make a machine in the likeness of a human mind"
phones = [pronouncing.phones_for_word(p)[0] for p in text.split()]
sum([pronouncing.syllable_count(p) for p in phones])

16

In a similar manner, we can get a list of stresses of a given word with `pronouncing.stresses_for_word()`

In [32]:
pronouncing.stresses_for_word('permit')

['01', '12']

Or we can pass the phone string to `pronouncing.stresses()` to get one stress string.

In [35]:
pronouncing.stresses(pronouncing.phones_for_word('university')[0])

'20100'

#### Exercise 1
Check the pronunciation, stresses and syllables of a word of your choice

---
#### Advance Search with Regular Expressions



 `pronouncing.search()` allows you to search the pronouncing dictionary for words whose pronunciation matches a particular pattern. For example, to find words that have within them the same sounds as the word "sea":

In [34]:
phones = pronouncing.phones_for_word("sea")[0]
pronouncing.search(phones)[:10] # get only the first 10 results

['ac',
 'acaena',
 'acc',
 'accede',
 'acceded',
 'accedes',
 'acceding',
 'acetic',
 'ach',
 'addressee']

Or even with stresses.

In [52]:
pronouncing.search_stresses("010")[:10]

['aardema',
 'ababa',
 'abadaka',
 'abadi',
 'abadie',
 'abalkin',
 'abalone',
 'abalones',
 'abalos',
 'abandon']

To make our searches more meaningful, we can combine the search with a vocabulary, it can either be a list of common English words, or words from a particular book/domain.

Below, we will reuse the common 10k English words from week 5

In [ ]:
!curl -L -O https://raw.githubusercontent.com/first20hours/google-10000-english/refs/heads/master/google-10000-english-usa-no-swears.txt

In [65]:
with open("../week5/google-10000-english-usa-no-swears.txt") as f:
    text = f.read()
common_words = text.split("\n")
print(len(common_words))

9885


In [79]:
stress_matches = pronouncing.search_stresses("0100")

def filtered_with_common_words(word_list):
    return [ word for word in word_list if word in common_words]

filtered_stress_matches = filtered_with_common_words(stress_matches)
print(len(filtered_stress_matches))
for w in filtered_stress_matches[:10]:
    print(w, pronouncing.stresses(pronouncing.phones_for_word(w)[0]))

557
aboriginal 20100
academy 0100
acceptable 0100
acceptable 0100
accessibility 200100
accessible 0100
accessories 0100
accessory 0100
accompanied 0100
accompanying 01000


By observing the returned result we can see that if we just give the word, or stress pattern, the returned results are all values that contain the pattern. What if we want an exact match, or a word that begins with xxx sound, or ends with y sound. In those cases, we can make use of something called <b>regular expression</b>.

In regular expression, we use the syntax `^` to mark the start of a string and `$` to mark the end of the string. So `^abc` will be any matches that starts with abs, and `abc$` will be any matches that ends with abc, `^abc$` will be exact matches of abc.

Try to modify the search pattern in the cell below, and observe the changes in the returned results.

In [80]:
stress_matches = pronouncing.search_stresses("^0100$")
filtered_stress_matches = filtered_with_common_words(stress_matches)
print(len(filtered_stress_matches))
for w in filtered_stress_matches[:10]:
    print(w, pronouncing.stresses(pronouncing.phones_for_word(w)[0]))

382
academy 0100
acceptable 0100
acceptable 0100
accessible 0100
accessories 0100
accessory 0100
accompanied 0100
accordingly 0100
accredited 0100
activities 0100


Another frequently used pattern is choices. `[abc]` means any single character from `abc`. So the following pattern `^[12]0[12]$` can be any result from 101,102,201,202.

In [108]:
stress_matches = pronouncing.search_stresses("^[12]0[12]$")
filtered_stress_matches = filtered_with_common_words(stress_matches)
print(len(filtered_stress_matches))
for w in filtered_stress_matches:
    if w.startswith("c"): # check words start with c
        print(w, pronouncing.stresses(pronouncing.phones_for_word(w)[0]))

375
cadillac 102
calculate 102
cameroon 102
cardiac 102
caroline 102
catalogue 102
celebrate 102
certified 102
chevrolet 201
chevrolet 201
cigarette 201
cigarettes 201
cingular 202
citysearch 102
classified 102
classifieds 102
commonwealth 102
companies 102
company 102
compromise 102
concentrate 102
constitute 102
constitutes 102
copyright 102
copyrights 102
customize 102
customized 102


We are only covering the most basic usage of regular expression here. It is a super powerful tool for string matching and text manipulation and can be used across different programming languages. You can learn more about its syntax [here](https://regex101.com/).

#### Exercise 2
Find common four-syllable words that begin with “K” sound.

In [128]:
print(pronouncing.phones_for_word("coach"))

def ex_1():
    result = []
    words = pronouncing.search("^K")
    for w in words:
        if pronouncing.syllable_count(pronouncing.phones_for_word(w)[0]) == 4 and w in common_words:
            print(w)
            result.append(w)
    print(len(result))
    return result

ex_1()


['K OW1 CH']
calculated
calculation
calculations
calculator
calculators
calibration
california
cambodia
canadian
cancellation
capacity
capacity
caribbean
caribbean
carolina
categories
category
characterized
cholesterol
cholesterol
coalition
collectible
collectibles
colombia
colonial
colorado
colorado
columbia
combination
combinations
comfortable
commentary
commissioner
commissioners
commodities
commodity
communicate
communities
communities
community
community
comparable
comparable
comparable
comparative
comparison
comparisons
compatible
compensation
competition
competitions
competitive
competitive
competitors
competitors
compilation
complexity
complexity
complicated
complications
composition
comprehensive
computation
concentration
concentrations
conceptual
conditional
conditioning
conferences
conferences
confidential
configuring
confirmation
congressional
connecticut
consecutive
consequences
consequently
consequently
conservation
conservative
considering
consistency
consistently
consor

['calculated',
 'calculation',
 'calculations',
 'calculator',
 'calculators',
 'calibration',
 'california',
 'cambodia',
 'canadian',
 'cancellation',
 'capacity',
 'capacity',
 'caribbean',
 'caribbean',
 'carolina',
 'categories',
 'category',
 'characterized',
 'cholesterol',
 'cholesterol',
 'coalition',
 'collectible',
 'collectibles',
 'colombia',
 'colonial',
 'colorado',
 'colorado',
 'columbia',
 'combination',
 'combinations',
 'comfortable',
 'commentary',
 'commissioner',
 'commissioners',
 'commodities',
 'commodity',
 'communicate',
 'communities',
 'communities',
 'community',
 'community',
 'comparable',
 'comparable',
 'comparable',
 'comparative',
 'comparison',
 'comparisons',
 'compatible',
 'compensation',
 'competition',
 'competitions',
 'competitive',
 'competitive',
 'competitors',
 'competitors',
 'compilation',
 'complexity',
 'complexity',
 'complicated',
 'complications',
 'composition',
 'comprehensive',
 'computation',
 'concentration',
 'concentrations

---


#### Rhymes

You can easily find potential rhymes of a given word by using `pronouncing.rhymes()`

In [23]:
pronouncing.rhymes("ocean")

['bocian',
 'commotion',
 'demotion',
 'devotion',
 'emotion',
 'hoeschen',
 'kocian',
 'laotian',
 'laotian',
 'locomotion',
 'lotion',
 'motion',
 'notion',
 'potion',
 'promotion',
 'promotion']

To check if one word rhymes with another one, you can use the same function with the `in` operator.

In [28]:
print("squeeze" in pronouncing.rhymes("cheese"))
print("maze" in pronouncing.rhymes("cheese"))


True
False


#### Exercise 3
Find nouns that rhyme with “rose.”

*Check the week4 notebook on spaCy for part-of-speech.

### Reference

[Pronouncing Tutorial](https://pronouncing.readthedocs.io/en/latest/tutorial.html)